# 03 · Régua "códigos mais comuns" na validação

Tarefa E4-T2 do [`docs/backlog.md`](../docs/backlog.md). Os números que saem daqui estão na secção 12 do [`docs/qualidade_dados.md`](../docs/qualidade_dados.md).

**Pergunta:** quanto acerta quem sugere sempre os mesmos 5 códigos, os mais frequentes no treino, sem ler o texto?

**Porque importa:** é o mínimo que qualquer modelo tem de bater. Se o TF-IDF (E4-H1) não ficar acima destes números, há um erro nos dados, no treino ou na avaliação. Esta lição serve para qualquer projeto de classificação: antes de treinar, mede-se a régua mais burra possível, porque um número sozinho ("acerta 40%") não diz se o modelo é bom.

**Como:**
- as etiquetas contam-se no treino **depois** da regra dos códigos raros (decisão 9), com o mesmo N que o modelo usa (por defeito 10), para a régua e o modelo escolherem entre as mesmas etiquetas;
- conta-se **por exemplo de treino** (um por texto), como a regra dos códigos raros. No fim do notebook compara-se com a régua que conta contratos (cada exemplo pesa o seu `n_contratos`), que ficou mais fácil de bater;
- avalia-se com a avaliação comum (`src/evaluation.py`, E4-T1), na validação (outubro de 2025). O teste não se abre.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import baselines as bl
import comparison as cmp
import cpv_data as cd
import evaluation as ev
from labels import DEFAULT_N_MIN, collapse_rare_codes

DATA = Path.cwd().parent / "data" / "processed"
treino = pd.read_parquet(DATA / "treino.parquet", columns=["codigo", "n_contratos"])
val = pd.read_parquet(DATA / "validacao.parquet", columns=["codigos", "visto_no_treino"])
print(f"treino: {len(treino):,} exemplos · validação: {len(val):,} contratos · N = {DEFAULT_N_MIN}")

treino: 478,384 exemplos · validação: 23,945 contratos · N = 10


## As 5 etiquetas da régua

O nome de cada código vem da coluna `CPV` de um ficheiro bruto (2024), só para se ler a tabela. Só se leem contratos publicados antes da validação, para não espreitar o período de teste.

In [2]:
top = bl.most_common_labels(treino["codigo"], n_min=DEFAULT_N_MIN)

label = collapse_rare_codes(treino["codigo"], n_min=DEFAULT_N_MIN)
counts = label.value_counts()
total = counts.sum()

cpv = cd.load_contracts(["CPV"], years=[2024], until=cd.VAL_START)["CPV"].dropna()
nomes = {}
for cel in cpv.drop_duplicates():
    for c, nome in re.findall(r"(\d{8})-\d\s*-\s*([^|\n]+)", cel):
        nomes.setdefault(c, nome.strip())

tabela = pd.DataFrame({
    "etiqueta": top,
    "nome": [nomes.get(c, "?") for c in top],
    "exemplos": [counts[c] for c in top],
    "% do treino": [counts[c] / total for c in top],
}, index=pd.RangeIndex(1, 6, name="posição"))
tabela.style.format({"exemplos": "{:,}", "% do treino": "{:.1%}"})

,etiqueta,nome,exemplos,% do treino
posição,,,,
1,33600000,Produtos farmacêuticos,"37,151",7.8%
2,33140000,Material médico de consumo,"22,540",4.7%
3,33690000,Medicamentos vários,"10,848",2.3%
4,33100000,Equipamento médico,"9,347",2.0%
5,98300000,Serviços diversos,"4,277",0.9%


As 5 juntas são esta parte dos exemplos de treino (depois da regra dos códigos raros):

In [3]:
print(f"{tabela['% do treino'].sum():.1%} dos exemplos de treino · {label.nunique():,} etiquetas distintas")

17.6% dos exemplos de treino · 3,000 etiquetas distintas


## Resultados na validação

Uma linha por medida (ver a regra de contagem no E4-T1):
- **codigo:** a sugestão é igual a um dos códigos do contrato (só contratos com código abaixo da divisão);
- **classe:** os mesmos 4 primeiros dígitos (só contratos com código ao nível da classe ou abaixo);
- **divisao:** os mesmos 2 primeiros dígitos (todos os contratos);
- **so_divisao:** os contratos que só têm códigos de divisão, à parte.

O grupo **texto novo** são os contratos cujo texto não aparece no treino. A régua não lê o texto, por isso não pode decorar: se der menos no texto novo, porquê? Responde a verificação 2, abaixo.

In [4]:
sugestoes = bl.suggest_most_common(top, val.index)
rel = ev.report(sugestoes, val["codigos"], val["visto_no_treino"])
rel.style.format({"contratos": "{:,}", "top1": "{:.1%}", "top5": "{:.1%}", "f1_macro": "{:.4f}"})

Medidas hierárquicas da 1.ª sugestão (só se mostram, não decidem):

In [5]:
ev.hierarchical_report(sugestoes, val["codigos"], val["visto_no_treino"]).style.format("{:.3f}")

,precisao_micro,recall_micro,f1_micro,precisao_macro,recall_macro,f1_macro
grupo,,,,,,
todos,0.222,0.149,0.179,0.039,0.018,0.025
texto novo,0.199,0.133,0.160,0.040,0.018,0.025


### Três verificações para ler a tabela

**1. Que sugestões acertam na classe?** Por cada uma das 5, quantos contratos (dos que contam na classe) acertam por causa dela.

In [6]:
cods = val["codigos"]
acertos = {c: ev.hits(bl.suggest_most_common([c] * 5, cods.index), cods, "classe", 1) for c in top}
pd.DataFrame({
    "contratos que acerta": {c: int(h.sum()) for c, h in acertos.items()},
    "% dos que contam na classe": {c: f"{h.mean():.1%}" for c, h in acertos.items()},
}).rename_axis("sugestão")

,contratos que acerta,% dos que contam na classe
sugestão,,
33600000,0,0.0%
33140000,1753,12.5%
33690000,438,3.1%
33100000,0,0.0%
98300000,0,0.0%


**2. Porque é que a régua acerta menos no texto novo, se não lê o texto?** Parte dos contratos de cada grupo cuja etiqueta (1.º código) é uma das 5 da régua.

In [7]:
etiqueta = cods.str.split().str[0]
(etiqueta.isin(top).groupby(val["visto_no_treino"].map({True: "texto já visto", False: "texto novo"}))
    .agg(["size", "mean"]).rename(columns={"size": "contratos", "mean": "etiqueta é uma das 5"})
    .style.format({"contratos": "{:,}", "etiqueta é uma das 5": "{:.1%}"}))

,contratos,etiqueta é uma das 5
visto_no_treino,,
texto já visto,"3,706",35.4%
texto novo,"20,239",19.4%


**3. Contratos só com divisão:** a linha `so_divisao` conta os contratos com **todos** os códigos ao nível da divisão; a secção 11 do `qualidade_dados.md` conta os contratos com a **etiqueta** ao nível da divisão.

In [8]:
so_div = cods.str.split().map(lambda cs: all(c.endswith("000000") for c in cs))
print(f"todos os códigos de divisão: {so_div.sum():,} · etiqueta de divisão: {etiqueta.str.endswith('000000').sum():,}")

todos os códigos de divisão: 4,039 · etiqueta de divisão: 4,051


## Muda com o N?

O spike E4-S1 vai escolher o N entre 5, 10 e 20. Se as 5 etiquetas forem as mesmas nos três N, a régua não precisa de voltar a calcular-se quando o E4-S1 escolher o N.

In [9]:
por_n = pd.DataFrame({f"N = {n}": bl.most_common_labels(treino["codigo"], n_min=n) for n in (5, 10, 20)},
                     index=pd.RangeIndex(1, 6, name="posição"))
por_n["nome (se mudar)"] = [nomes.get(r["N = 5"], "?") if r.nunique() > 1 else "" for _, r in por_n.iterrows()]
por_n

,N = 5,N = 10,N = 20,nome (se mudar)
posição,,,,
1,33600000,33600000,33600000,
2,33140000,33140000,33140000,
3,33690000,33690000,33690000,
4,33100000,33100000,33100000,
5,98300000,98300000,98300000,


## Contar exemplos ou contratos?

O treino tem um exemplo por texto (decisão 6), e o `n_contratos` diz quantos contratos tinham esse texto. "Mais comum" pode querer dizer o código com mais **exemplos** (textos diferentes) ou com mais **contratos**. Primeiro escolheu-se contar contratos, porque a validação tem todos os contratos com as repetições; as tabelas abaixo mostraram que essa régua fica mais fácil de bater. Ficou a que conta exemplos, a mais exigente das duas (decisão 10 do `qualidade_dados.md`).

In [10]:
top_ct = bl.most_common_labels(treino["codigo"], treino["n_contratos"], n_min=DEFAULT_N_MIN)
pd.DataFrame({"por exemplo (a régua)": top, "por contrato": top_ct}, index=pd.RangeIndex(1, 6, name="posição"))

,por exemplo (a régua),por contrato
posição,,
1,33600000,33600000
2,33140000,33140000
3,33690000,33100000
4,33100000,33690000
5,98300000,15800000


In [11]:
rel_ct = ev.report(bl.suggest_most_common(top_ct, val.index), val["codigos"], val["visto_no_treino"])
lado = pd.concat({"por exemplo (a régua)": rel["top5"], "por contrato": rel_ct["top5"]}, axis=1)
lado.loc[(slice(None), ["codigo", "classe"]), :].style.format("{:.1%}")

A diferença é pequena. Conta, ou é acaso? Bootstrap emparelhado do E4-T1 (`src/comparison.py`): 1 000 amostras dos contratos de validação, as mesmas para as duas réguas. A diferença conta se uma ganhar em pelo menos 95% das amostras. A = por exemplo (a régua), B = por contrato.

In [12]:
sug_ct = bl.suggest_most_common(top_ct, val.index)
novo = ~val["visto_no_treino"]
boot = pd.concat({
    "todos": cmp.compare(sugestoes, sug_ct, val["codigos"]),
    "texto novo": cmp.compare(sugestoes[novo], sug_ct[novo], val["codigos"][novo]),
}, names=["grupo"])
boot = boot.loc[(slice(None), ["codigo", "classe"], "top5"), ["a", "b", "diferenca", "a_melhor", "b_melhor", "conta"]]
boot.style.format({"a": "{:.2%}", "b": "{:.2%}", "diferenca": "{:+.2%}", "a_melhor": "{:.1%}", "b_melhor": "{:.1%}"})

,,,a,b,diferenca,a_melhor,b_melhor,conta
grupo,medida,metrica,,,,,,
todos,codigo,top5,26.32%,25.98%,+0.35%,100.0%,0.0%,True
texto novo,codigo,top5,23.72%,23.25%,+0.46%,100.0%,0.0%,True
todos,classe,top5,15.60%,15.60%,+0.00%,0.0%,0.0%,False
texto novo,classe,top5,14.84%,14.84%,+0.00%,0.0%,0.0%,False


## Como ler

As conclusões estão escritas na secção 12 do [`qualidade_dados.md`](../docs/qualidade_dados.md), com os números das tabelas acima.